In [1]:
import sqlite3, os, time
from pathlib import Path

In [2]:
from app.conventions import DBErrorKind, OutcomeKind, BackendKeys
from app.db.sqlite_connection import unit_of_work
from app.services.naming import string_normalize
from app.db.sqlite_errors import translate_sqlite_error

In [3]:
from app.config import DB_PATH

In [4]:

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

In [42]:
def attempt(conn, label, sql, params):
    start = time.perf_counter()
    try:
        conn.execute(sql, params)
        print(f"{label:<28} Statement Okay after {time.perf_counter() - start: .2f}s")
        return True 
    except sqlite3.Error as err:
        print(f"{label:<28} Failed with {type(err).__name__} after {time.perf_counter() - start: .2f}s")
        print("| errorname: ", getattr(err, "sqlite_errorname", None),
              "\n| errorcode: ", getattr(err, "sqlite_errorcode", None),
              "\n| errormessage: ", err)
        return False 

def rows(conn):
    result = conn.execute("SELECT * FROM clients ORDER BY id").fetchall()
    return result

def show(conn):
    clients = rows(conn)
    notes = conn.execute("SELECT id, client_id, note FROM client_note ORDER BY id").fetchall()
    print(f"clients: {clients}\nnotes: {notes}\n")

In [6]:
setup_sql = Path('./app/db/schema.sql').read_text()

In [7]:
with unit_of_work(DB_PATH) as conn:
    conn.executescript(setup_sql)
    show(conn)

clients: []
notes: []



In [8]:
with unit_of_work(DB_PATH) as conn:
    conn.executemany("INSERT INTO clients (name, name_standardized, modified_by) VALUES (?, ?, ?)",
                     [("Farm A", string_normalize("Farm A"), "Test User 1"),
                     ("Farm B", string_normalize("Farm B"), "Test User 2")])
    idx = conn.execute("SELECT id FROM clients WHERE name = ?", ("Farm A", )).fetchone()[0]
    conn.execute("INSERT INTO client_note (client_id, note) VALUES (?, ?)", (idx, "test note"))
    show(conn)

clients: [(1, 'Farm A', 'farm_a', 'Test User 1'), (2, 'Farm B', 'farm_b', 'Test User 2')]
notes: [(1, 1, 'test note')]



In [9]:
with unit_of_work(DB_PATH) as conn:
    conn.execute("INSERT INTO clients (name, name_standardized, modified_by) VALUES (?, ?, ?)", ("Farm     C", string_normalize("Farm     C"), "Test User 3"))
    show(conn)
    

clients: [(1, 'Farm A', 'farm_a', 'Test User 1'), (2, 'Farm B', 'farm_b', 'Test User 2'), (3, 'Farm     C', 'farm_c', 'Test User 3')]
notes: [(1, 1, 'test note')]



## services